# Hands-On 8: Build Supervisor Multi-Agent System

A supervisor agent routes tasks to specialized agents (research, analysis, report) and combines their outputs.

#### Imports

We import `TypedDict` for state and `Literal` for routing return types, plus `StateGraph`, `START`, `END` from LangGraph.

In [1]:
from typing import TypedDict, Literal

import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI

from langgraph.graph import StateGraph, START, END

In [2]:
load_dotenv(r"D:\Agent_Flow\.env", override=True)

print("OpenRouter API key loaded:", bool(os.getenv("OPENROUTER_API_KEY")))

OpenRouter API key loaded: True


In [3]:
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0,
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

print("OpenRouter LLM configured successfully")

OpenRouter LLM configured successfully


In [4]:
response = llm.invoke(
    "Explain artificial intelligence in one sentence."
)

print(response.content)

Artificial intelligence is the simulation of human intelligence processes by machines, particularly computer systems, enabling them to perform tasks such as learning, reasoning, problem-solving, and understanding natural language.


#### Define Shared State

All agents read and write to this shared state containing topic, research, analysis, report, and status fields.

In [5]:
class AgentState(TypedDict):
    topic: str
    research: str
    analysis: str
    report: str
    status: str

#### Create Specialized Agents

We create three agents: `research_agent` collects data, `analysis_agent` interprets it, and `report_agent` writes the final report.

In [6]:
def research_agent(state: AgentState):
    print("🔍 Research agent working...")

    prompt = f"""
You are the Research Agent.

Research the following topic:

{state['topic']}

Provide useful research findings including:
- Definition
- Key concepts
- Important developments
- Current applications
- Important considerations

Keep the response concise but informative.
"""

    response = llm.invoke(prompt)

    return {
        "research": response.content,
        "status": "researched"
    }

#### Create Supervisor

The supervisor is a routing function that examines the status field and decides which agent should run next.

In [7]:
def analysis_agent(state: AgentState):
    print("📊 Analysis agent working...")

    prompt = f"""
You are the Analysis Agent.

Analyze the research below.

Topic:
{state['topic']}

Research:
{state['research']}

Provide:
- Key insights
- Strengths
- Challenges
- Opportunities
- Risks
- Practical implications

Base your analysis on the research provided.
"""

    response = llm.invoke(prompt)

    return {
        "analysis": response.content,
        "status": "analyzed"
    }

## # Create report agent

In [9]:
def report_agent(state: AgentState):
    print("📝 Report agent working...")

    prompt = f"""
You are the Report Agent.

Create a final professional report using the information below.

Topic:
{state['topic']}

Research:
{state['research']}

Analysis:
{state['analysis']}

Create a concise report with these sections:

# Report on {state['topic']}

## Executive Summary

## Research Findings

## Analysis

## Key Insights

## Conclusion
"""

    response = llm.invoke(prompt)

    return {
        "report": response.content,
        "status": "completed"
    }

## Create Supervisor

In [10]:
def supervisor(
    state: AgentState
) -> Literal[
    "research_agent",
    "analysis_agent",
    "report_agent",
    "__end__"
]:
    
    status = state.get("status", "")

    if status == "":
        return "research_agent"

    elif status == "researched":
        return "analysis_agent"

    elif status == "analyzed":
        return "report_agent"

    return "__end__"

#### Build Graph

We connect the agents in sequence using conditional edges via the supervisor routing logic.

In [11]:
graph = StateGraph(AgentState)

graph.add_node(
    "research_agent",
    research_agent
)

graph.add_node(
    "analysis_agent",
    analysis_agent
)

graph.add_node(
    "report_agent",
    report_agent
)

graph.add_edge(
    START,
    "research_agent"
)

print("Supervisor graph nodes created successfully")

Supervisor graph nodes created successfully


#### Route Tasks

Invoke the system with a topic. The supervisor routes through research → analysis → report, updating shared state at each step.

In [12]:
graph.add_conditional_edges(
    "research_agent",
    supervisor,
    {
        "analysis_agent": "analysis_agent",
        "report_agent": "report_agent",
        "__end__": END
    }
)

### Route Analysis → Report

In [13]:
graph.add_conditional_edges(
    "analysis_agent",
    supervisor,
    {
        "analysis_agent": "analysis_agent",
        "report_agent": "report_agent",
        "__end__": END
    }
)

### Route Report → End

In [14]:
graph.add_conditional_edges(
    "report_agent",
    supervisor,
    {
        "analysis_agent": "analysis_agent",
        "report_agent": "report_agent",
        "__end__": END
    }
)

print("Supervisor routing configured successfully")

Supervisor routing configured successfully


### Compile Multi-Agent System

In [15]:
multi_agent = graph.compile()

print("Multi-agent supervisor compiled successfully")

Multi-agent supervisor compiled successfully


### Run First Task

Use the notebook's Artificial Intelligence example:

In [16]:
result = multi_agent.invoke({
    "topic": "Artificial Intelligence",
    "research": "",
    "analysis": "",
    "report": "",
    "status": ""
})

print("Workflow completed.")
print("Status:", result["status"])

🔍 Research agent working...
📊 Analysis agent working...
📝 Report agent working...
Workflow completed.
Status: completed


In [17]:
print("=== RESEARCH OUTPUT ===")
print(result["research"])

=== RESEARCH OUTPUT ===
### Artificial Intelligence (AI)

#### Definition
Artificial Intelligence (AI) refers to the simulation of human intelligence processes by machines, particularly computer systems. These processes include learning (the acquisition of information and rules for using it), reasoning (using rules to reach approximate or definite conclusions), and self-correction.

#### Key Concepts
1. **Machine Learning (ML)**: A subset of AI that enables systems to learn from data and improve their performance over time without being explicitly programmed.
2. **Deep Learning**: A specialized form of ML that uses neural networks with many layers (deep networks) to analyze various factors of data.
3. **Natural Language Processing (NLP)**: The ability of a computer to understand, interpret, and generate human language.
4. **Computer Vision**: The field of AI that enables machines to interpret and make decisions based on visual data from the world.
5. **Robotics**: The integration of AI

In [18]:
print("=== ANALYSIS OUTPUT ===")
print(result["analysis"])

=== ANALYSIS OUTPUT ===
### Analysis of Artificial Intelligence Research

#### Key Insights
- **Definition and Scope**: AI encompasses a broad range of technologies that simulate human intelligence, including learning, reasoning, and self-correction.
- **Subfields**: Key areas within AI such as Machine Learning, Deep Learning, Natural Language Processing, Computer Vision, and Robotics highlight the diversity and specialization within the field.
- **Historical Context**: The evolution of AI from the Turing Test in the 1950s to the development of large language models in the 2020s illustrates significant technological advancements and shifts in focus.

#### Strengths
- **Versatility**: AI's applications span multiple industries, including healthcare, finance, transportation, customer service, and manufacturing, showcasing its adaptability and potential for impact.
- **Efficiency and Accuracy**: AI systems can process vast amounts of data quickly and with high accuracy, leading to improve

#### Combine Agent Outputs

All agents write to shared state. We inspect the final report and status.

In [19]:
print("=" * 60)
print("FINAL MULTI-AGENT STATE")
print("=" * 60)

print("\nTopic:")
print(result["topic"])

print("\nResearch:")
print(result["research"])

print("\nAnalysis:")
print(result["analysis"])

print("\nReport:")
print(result["report"])

print("\nStatus:")
print(result["status"])

FINAL MULTI-AGENT STATE

Topic:
Artificial Intelligence

Research:
### Artificial Intelligence (AI)

#### Definition
Artificial Intelligence (AI) refers to the simulation of human intelligence processes by machines, particularly computer systems. These processes include learning (the acquisition of information and rules for using it), reasoning (using rules to reach approximate or definite conclusions), and self-correction.

#### Key Concepts
1. **Machine Learning (ML)**: A subset of AI that enables systems to learn from data and improve their performance over time without being explicitly programmed.
2. **Deep Learning**: A specialized form of ML that uses neural networks with many layers (deep networks) to analyze various factors of data.
3. **Natural Language Processing (NLP)**: The ability of a computer to understand, interpret, and generate human language.
4. **Computer Vision**: The field of AI that enables machines to interpret and make decisions based on visual data from the wo

### Test Another Topic
Blockchain

In [20]:
result2 = multi_agent.invoke({
    "topic": "Blockchain",
    "research": "",
    "analysis": "",
    "report": "",
    "status": ""
})

print("Status:", result2["status"])

print("\n=== FINAL REPORT ===")
print(result2["report"])

🔍 Research agent working...
📊 Analysis agent working...
📝 Report agent working...
Status: completed

=== FINAL REPORT ===
# Report on Blockchain

## Executive Summary
Blockchain technology has emerged as a transformative force across various sectors, providing a decentralized and secure method for recording transactions. This report synthesizes research findings and analysis of blockchain, highlighting its definition, key concepts, important developments, current applications, and the challenges and opportunities it presents. While blockchain offers significant advantages in terms of transparency, security, and efficiency, it also faces hurdles such as scalability, energy consumption, and regulatory uncertainties that must be addressed for broader adoption.

## Research Findings

### Definition
Blockchain is a decentralized, distributed ledger technology that securely records transactions across multiple computers, ensuring that recorded transactions cannot be altered retroactively wit

### Test a More Complex Topic
— Generative AI

In [21]:
result3 = multi_agent.invoke({
    "topic": "Generative AI in Enterprise Applications",
    "research": "",
    "analysis": "",
    "report": "",
    "status": ""
})

print("Status:", result3["status"])

print("\n=== FINAL REPORT ===")
print(result3["report"])

🔍 Research agent working...
📊 Analysis agent working...
📝 Report agent working...
Status: completed

=== FINAL REPORT ===
# Report on Generative AI in Enterprise Applications

## Executive Summary
Generative AI is revolutionizing enterprise applications by automating processes, enhancing creativity, and improving decision-making. This report explores the definition, key concepts, current applications, and challenges associated with generative AI in enterprises. It highlights the importance of ethical considerations, data privacy, and integration challenges while also identifying opportunities for innovation and market expansion.

## Research Findings
Generative AI refers to algorithms capable of creating new content—text, images, audio, or code—based on training data. Key concepts include machine learning models such as Generative Adversarial Networks (GANs) and transformer models like GPT and BERT. Current applications span content creation, customer support, product design, and data 

#### Generate Final Report (Markdown)

Display the final report with its research and analysis sections nicely formatted.

In [22]:
print("=== HANDS-ON 8 COMPLETE ===")

print("✓ OpenRouter API configured")
print("✓ Supervisor created")
print("✓ Research agent created")
print("✓ Analysis agent created")
print("✓ Report agent created")
print("✓ Tasks routed by supervisor")
print("✓ Shared state passed between agents")
print("✓ Agent outputs combined")
print("✓ Final report generated")
print("✓ Multiple topics tested")

=== HANDS-ON 8 COMPLETE ===
✓ OpenRouter API configured
✓ Supervisor created
✓ Research agent created
✓ Analysis agent created
✓ Report agent created
✓ Tasks routed by supervisor
✓ Shared state passed between agents
✓ Agent outputs combined
✓ Final report generated
✓ Multiple topics tested


#### Test: Different Topic

Re-run the same multi-agent system with a new topic to generate a different report.

#### Summary

Multi-agent systems divide complex work among specialists. Each agent writes to shared state, and the supervisor routes the work flow.